### Make the folders

In [1]:
import os
for f in ["../src", "../models", "../app"]:
    os.makedirs(f, exist_ok=True)
print("Folders ready")

Folders ready


### Write the tool file (the first line must be exactly %%writefile ../src/olist_tools.py)

In [2]:
%%writefile ../src/olist_tools.py
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin


class Winsorizer(BaseEstimator, TransformerMixin):
    """Caps extreme values. Limits are learned from the training data only."""

    def __init__(self, lower=0.01, upper=0.99):
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        self.low_ = np.nanquantile(X, self.lower, axis=0)
        self.high_ = np.nanquantile(X, self.upper, axis=0)
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        return np.clip(X, self.low_, self.high_)

    def get_feature_names_out(self, input_features=None):
        return np.asarray(input_features, dtype=object)

Writing ../src/olist_tools.py


### Setup (brings your saved work into this notebook)

In [3]:
import sys, json
import numpy as np
import pandas as pd
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score

sys.path.insert(0, "../src")
from olist_tools import Winsorizer

print("Saved files found:", os.path.exists("../data/full_v2.pkl"), os.path.exists("../data/feature_list.json"))

full = pd.read_pickle("../data/full_v2.pkl")
with open("../data/feature_list.json") as f:
    cfg = json.load(f)
numeric_features = cfg["numeric"]
categorical_features = cfg["categorical"]

chosen_num = [c for c in numeric_features if c != "purchase_month"] + ["seller_90d_rate", "route_60d_rate"]
chosen_cols = chosen_num + categorical_features

pool = full[full["order_purchase_timestamp"] < "2018-05-01"]
test2 = full[full["order_purchase_timestamp"] >= "2018-05-01"]
y_t = test2["is_late"].values

numeric_pipe = Pipeline([
    ("cap", Winsorizer()),
    ("fill", SimpleImputer(strategy="median", add_indicator=True)),
    ("scale", StandardScaler()),
])
categorical_pipe = Pipeline([
    ("fill", SimpleImputer(strategy="constant", fill_value="unknown")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=100)),
])

def make_final():
    prep = ColumnTransformer([
        ("num", numeric_pipe, chosen_num),
        ("cat", categorical_pipe, categorical_features),
    ])
    return Pipeline([("prep", prep), ("model", LogisticRegression(max_iter=2000))])

print(full.shape, len(pool), len(test2))

Saved files found: True True
(96203, 57) 70851 25352


### train the final model and check it

In [6]:
final_model = make_final()
final_model.fit(pool[chosen_cols], pool["is_late"])

p_test = final_model.predict_proba(test2[chosen_cols])[:, 1]
print("PR-AUC :", round(average_precision_score(y_t, p_test), 4))
print("ROC-AUC:", round(roc_auc_score(y_t, p_test), 4))

PR-AUC : 0.0916
ROC-AUC: 0.6648


### risk bands, settings, and save

In [7]:
p_pool = final_model.predict_proba(pool[chosen_cols])[:, 1]
cut_medium, cut_high = np.quantile(p_pool, [0.70, 0.90])

def band(p):
    return np.where(p >= cut_high, "High", np.where(p >= cut_medium, "Medium", "Low"))

bt = pd.DataFrame({"band": band(p_test), "late": y_t}).groupby("band")["late"].agg(
    orders="size", late_rate="mean"
)
bt["late_rate_pct"] = (bt["late_rate"] * 100).round(2)
bt["times_average"] = (bt["late_rate"] / y_t.mean()).round(2)
bt = bt.loc[["Low", "Medium", "High"]]
print(bt)

defaults = {}
for c in chosen_cols:
    if c in chosen_num:
        defaults[c] = float(pool[c].median())
    else:
        defaults[c] = str(pool[c].mode()[0])

choices = {c: sorted(pool[c].dropna().astype(str).unique().tolist()) for c in categorical_features}

settings = {
    "numeric_clues": chosen_num,
    "category_clues": categorical_features,
    "cut_medium": float(cut_medium),
    "cut_high": float(cut_high),
    "train_period": "2017-01-01 to 2018-04-30",
    "test_period": "2018-05-01 to 2018-08-29",
    "test_late_rate": float(y_t.mean()),
    "band_stats": bt.reset_index()[["band", "orders", "late_rate_pct", "times_average"]].to_dict("records"),
    "defaults": defaults,
    "choices": choices,
}

joblib.dump(final_model, "../models/late_delivery_model.joblib")
with open("../models/settings.json", "w") as f:
    json.dump(settings, f, indent=2)
print("Saved model and settings")

        orders  late_rate  late_rate_pct  times_average
band                                                   
Low      14637   0.028763           2.88           0.65
Medium    6015   0.046384           4.64           1.05
High      4700   0.088298           8.83           2.01
Saved model and settings


### check that the saved model loads

In [8]:
loaded = joblib.load("../models/late_delivery_model.joblib")
print("Loaded model works?", np.allclose(loaded.predict_proba(test2[chosen_cols])[:, 1], p_test))

Loaded model works? True


# The app file

In [1]:
%%writefile ../app/app.py
import json, sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import streamlit as st

BASE = Path(__file__).resolve().parent.parent
sys.path.insert(0, str(BASE / "src"))

st.set_page_config(page_title="Olist Late-Delivery Risk")


@st.cache_resource
def load():
    model = joblib.load(BASE / "models" / "late_delivery_model.joblib")
    with open(BASE / "models" / "settings.json") as f:
        settings = json.load(f)
    return model, settings


model, S = load()
D = S["defaults"]

FRIENDLY = {
    "distance_km": "Distance and speed needed to meet the promise",
    "km_per_promised_day": "Distance and speed needed to meet the promise",
    "promise_days": "Promised delivery time",
    "courier_window_days": "Time left for the courier",
    "days_to_last_limit": "Seller hand-over deadline",
    "days_to_first_limit": "Seller hand-over deadline",
    "same_state": "Seller and customer in the same state",
    "customer_state": "Customer state",
    "seller_state": "Seller state",
    "main_category": "Product category",
    "main_payment_type": "Payment type",
    "total_weight_g": "Parcel weight",
    "max_weight_g": "Parcel weight",
    "total_volume_cm3": "Parcel size",
    "total_price": "Price and delivery charge",
    "total_freight": "Price and delivery charge",
    "freight_ratio": "Price and delivery charge",
    "total_payment": "Price and delivery charge",
    "route_60d_rate": "Route's recent late rate",
    "seller_90d_rate": "Seller's recent late rate",
    "n_items": "Number of items",
    "n_products": "Number of items",
}


def group_name(col):
    return FRIENDLY.get(col, col.replace("_", " "))


def to_dense(x):
    return x.toarray() if hasattr(x, "toarray") else np.asarray(x)


def pick(label, key):
    options = S["choices"][key]
    default = D[key]
    index = options.index(default) if default in options else 0
    return st.selectbox(label, options, index=index)


st.title("Olist Late-Delivery Early Warning")
st.caption(
    "Demo built on Olist data (Brazil, 2017-2018). It shows patterns, not causes, "
    "and it has not been tested as a way to prevent delays."
)

left, right = st.columns(2)
with left:
    customer_state = pick("Customer state", "customer_state")
    seller_state = pick("Seller state", "seller_state")
    category = pick("Product category", "main_category")
    payment = pick("Payment type", "main_payment_type")
    n_items = st.number_input("Number of items", min_value=1, max_value=10, value=1)
with right:
    distance = st.slider("Distance seller to customer (km)", 0, 3500, int(D["distance_km"]), step=10)
    promise = st.slider("Promised delivery time (days)", 3, 60, int(min(max(D["promise_days"], 3), 60)))
    handover = st.slider(
        "Seller hand-over deadline (days after purchase)",
        1.0, 30.0, float(min(max(round(D["days_to_last_limit"], 1), 1.0), 30.0)), step=0.5,
    )
    weight = st.number_input("Total parcel weight (grams)", min_value=0.0, value=float(D["total_weight_g"]), step=100.0)
    price = st.number_input("Total product price", min_value=0.0, value=float(D["total_price"]), step=10.0)
    freight = st.number_input("Delivery charge (freight)", min_value=0.0, value=float(D["total_freight"]), step=1.0)

with st.expander("Advanced: recent late rates from past delivered orders"):
    seller_rate = st.slider("Seller's late rate, last 90 days (%)", 0.0, 50.0,
                            float(min(round(D["seller_90d_rate"] * 100, 1), 50.0)), step=0.5)
    route_rate = st.slider("Route's late rate, last 60 days (%)", 0.0, 50.0,
                           float(min(round(D["route_60d_rate"] * 100, 1), 50.0)), step=0.5)

row = dict(D)
row.update({
    "customer_state": customer_state,
    "seller_state": seller_state,
    "main_category": category,
    "main_payment_type": payment,
    "n_items": float(n_items),
    "n_products": float(n_items),
    "distance_km": float(distance),
    "promise_days": float(promise),
    "km_per_promised_day": distance / promise,
    "days_to_last_limit": handover,
    "days_to_first_limit": handover,
    "courier_window_days": promise - handover,
    "same_state": float(customer_state == seller_state),
    "total_weight_g": weight,
    "max_weight_g": weight / n_items,
    "total_price": price,
    "total_freight": freight,
    "freight_ratio": freight / (price + 1),
    "total_payment": price + freight,
    "seller_90d_rate": seller_rate / 100,
    "route_60d_rate": route_rate / 100,
})

cols = S["numeric_clues"] + S["category_clues"]
X = pd.DataFrame([row])[cols]
X0 = pd.DataFrame([D])[cols]

score = float(model.predict_proba(X)[0, 1])
if score >= S["cut_high"]:
    band = "High"
elif score >= S["cut_medium"]:
    band = "Medium"
else:
    band = "Low"
stats = {b["band"]: b for b in S["band_stats"]}[band]

st.divider()
st.subheader("Risk band: " + band)
text = (
    f"In the test months (May to Aug 2018), orders in this band were late "
    f"{stats['late_rate_pct']}% of the time, about {stats['times_average']} times the "
    f"overall average of {S['test_late_rate'] * 100:.1f}%."
)
if band == "High":
    st.warning(text)
    st.write("Suggested action: consider a manual check. Confirm that the seller hands the parcel "
             "to the courier on time and that the promised date is realistic. "
             "We have not tested whether this action prevents delays.")
elif band == "Medium":
    st.info(text)
    st.write("Suggested action: keep normal monitoring.")
else:
    st.success(text)
    st.write("Suggested action: normal handling. Low risk does not mean no risk, "
             "because some late orders are also found in this band.")

prep = model.named_steps["prep"]
coef = model.named_steps["model"].coef_[0]
names = prep.get_feature_names_out()
delta = to_dense(prep.transform(X))[0] - to_dense(prep.transform(X0))[0]
contrib = coef * delta

groups = {}
for name, value in zip(names, contrib):
    clean = name.split("__", 1)[1].replace("missingindicator_", "")
    base = clean
    for c in S["category_clues"]:
        if clean.startswith(c + "_"):
            base = c
            break
    label = group_name(base)
    groups[label] = groups.get(label, 0.0) + float(value)

top = sorted(groups.items(), key=lambda kv: abs(kv[1]), reverse=True)[:4]
top = [kv for kv in top if abs(kv[1]) > 0.01]

st.markdown("**What moves this score compared with a typical order**")
if not top:
    st.write("This order looks like a typical order, so no factor stands out.")
else:
    lines = []
    for label, value in top:
        lines.append("- " + label + ": " + ("raises risk" if value > 0 else "lowers risk"))
    st.markdown("\n".join(lines))

with st.expander("Technical details"):
    st.write(f"Raw model score: {score:.3f} (High starts at {S['cut_high']:.3f}, "
             f"Medium at {S['cut_medium']:.3f}).")
    st.write("The raw score is not a true percentage. In the newest months the model's "
             "scores were higher than the real late rates, so use the band instead.")
    st.write("Model: logistic regression trained on orders from " + S["train_period"] +
             " and tested on " + S["test_period"] + ".")

st.caption("Limits: one country, two years of data, purchase-time information only. "
           "Clues that are not shown here are filled with typical values.")

Writing ../app/app.py


In [2]:
import sys, os
print("STEP 1 (install the app tool, once):")
print(f'"{sys.executable}" -m pip install streamlit')
print()
print("STEP 2 (start the app):")
print(f'"{sys.executable}" -m streamlit run "{os.path.abspath("../app/app.py")}"')

STEP 1 (install the app tool, once):
"C:\Users\r9737\anaconda3\envs\olist\python.exe" -m pip install streamlit

STEP 2 (start the app):
"C:\Users\r9737\anaconda3\envs\olist\python.exe" -m streamlit run "C:\Users\r9737\OneDrive\New folder (3)\OLIST\app\app.py"


### ENGLISH TRANSLATION 

In [4]:
import json
import pandas as pd

# 1. Build the translation list
tr = pd.read_csv("../data/product_category_name_translation.csv", encoding="utf-8-sig")
names = {
    str(a): str(b).replace("_", " ").title()
    for a, b in zip(tr.iloc[:, 0], tr.iloc[:, 1])
}
names["pc_gamer"] = "PC Gamer"
names["portateis_cozinha_e_preparadores_de_alimentos"] = "Portable Kitchen And Food Preparers"

with open("../models/category_translation.json", "w", encoding="utf-8") as f:
    json.dump(names, f, indent=2, ensure_ascii=False)

# 2. Update the app file
path = "../app/app.py"
code = open(path, encoding="utf-8").read()

old = '''def pick(label, key):
    options = S["choices"][key]
    default = D[key]
    index = options.index(default) if default in options else 0
    return st.selectbox(label, options, index=index)
'''

new = '''try:
    with open(BASE / "models" / "category_translation.json", encoding="utf-8") as f:
        NAMES = json.load(f)
except Exception:
    NAMES = {}


def show_name(value):
    return NAMES.get(value, str(value).replace("_", " "))


def pick(label, key):
    options = S["choices"][key]
    default = D[key]
    index = options.index(default) if default in options else 0
    return st.selectbox(label, options, index=index, format_func=show_name)
'''

assert old in code, "Could not find the old text in app.py"
open(path, "w", encoding="utf-8").write(code.replace(old, new))
print("Done:", len(names), "category names translated")

AssertionError: Could not find the old text in app.py

In [5]:
import json

with open("../models/category_translation.json", encoding="utf-8") as f:
    names = json.load(f)

states = {
    "AC": "Acre", "AL": "Alagoas", "AP": "Amapá", "AM": "Amazonas",
    "BA": "Bahia", "CE": "Ceará", "DF": "Distrito Federal", "ES": "Espírito Santo",
    "GO": "Goiás", "MA": "Maranhão", "MT": "Mato Grosso", "MS": "Mato Grosso do Sul",
    "MG": "Minas Gerais", "PA": "Pará", "PB": "Paraíba", "PR": "Paraná",
    "PE": "Pernambuco", "PI": "Piauí", "RJ": "Rio de Janeiro",
    "RN": "Rio Grande do Norte", "RS": "Rio Grande do Sul", "RO": "Rondônia",
    "RR": "Roraima", "SC": "Santa Catarina", "SP": "São Paulo",
    "SE": "Sergipe", "TO": "Tocantins",
}
for code, full in states.items():
    names[code] = f"{full} ({code})"

with open("../models/category_translation.json", "w", encoding="utf-8") as f:
    json.dump(names, f, indent=2, ensure_ascii=False)

print("Done:", len(names), "names saved")

Done: 100 names saved


# Folders and Requirements.txt

In [6]:
import os
from importlib.metadata import version

os.makedirs("../tests", exist_ok=True)

pkgs = ["pandas", "numpy", "scikit-learn", "joblib", "streamlit"]
lines = [f"{p}=={version(p)}" for p in pkgs]
with open("../requirements.txt", "w") as f:
    f.write("\n".join(lines) + "\n")
print("\n".join(lines))

pandas==3.0.6
numpy==2.4.6
scikit-learn==1.9.1
joblib==1.6.0
streamlit==1.66.1


# README

In [7]:
%%writefile ../README.md
# Olist Late-Delivery Early Warning

Predicts, at the moment an order is placed, how likely it is to arrive after its promised delivery date. Built on the public Olist e-commerce dataset (Brazil, 2016-2018) with a Streamlit demo.

## Why this project

Late deliveries upset customers. An operations team cannot check every order, so the model ranks orders by risk and the team checks the riskiest ones first.

## Data

- Source: Brazilian E-Commerce Public Dataset by Olist (Kaggle), 9 CSV tables. The data is not stored in this repository. Check the dataset license before reusing it.
- 99,441 orders. 96,470 were delivered with valid dates and form the working group.
- Target: late = the order arrived on a later calendar day than the promised day. Overall late rate: 6.77%.
- Reviews were used only for the business story (late orders average 2.3 stars, on-time orders 4.3). They are never a model input, because they are written after delivery.

## Method

1. Audited all tables: missing values, duplicates, key uniqueness, impossible dates.
2. Built one row per order. One-to-many tables (items, payments, geolocation) were summarised before joining, and the row count was checked after every join.
3. Used only clues known when the order is placed: promised days, distance, distance per promised day, seller hand-over deadline, parcel weight and size, price and delivery charge, customer and seller state, product category, payment type, and the recent late rate of the seller and the route. Recent late rates use only orders already delivered before the purchase time (checked with a test).
4. Split by time, not randomly. Learning period: Jan 2017 - Apr 2018 (70,851 orders; Jan-Dec 2017 for training and Jan-Apr 2018 for validation). Test period: May - Aug 2018 (25,352 orders), used once. 267 orders from 2016 were left out because there were too few.
5. Compared a lazy baseline, logistic regression, random forest and LightGBM on the validation period only. The final model is logistic regression, retrained on the whole learning period.

## Results

Validation (Jan - Apr 2018, late rate 10.8%):

| Model | PR-AUC | ROC-AUC |
|---|---|---|
| Lazy baseline | 0.108 | 0.500 |
| Logistic regression (final clues) | 0.234 | 0.695 |
| LightGBM, simple (same clues) | 0.231 | 0.675 |

Random forest scored lower (PR-AUC 0.204, ROC-AUC 0.668) on an earlier clue set.

Held-out test (May - Aug 2018, late rate 4.4%):

- ROC-AUC 0.665 and PR-AUC 0.092 (a lazy model gets 0.044).
- The riskiest 5% of orders were late about 3.1 times as often as average and contained 15.7% of all late orders.
- The riskiest 10% were late about 2.6 times as often as average and contained 25.9% of all late orders.

Risk bands in the app (test period):

| Band | Orders | Really late | Compared with average |
|---|---|---|---|
| Low | 14,637 | 2.88% | 0.65x |
| Medium | 6,015 | 4.64% | 1.05x |
| High | 4,700 | 8.83% | 2.01x |

## What the model learned

The most important clue is how demanding the promise is (distance divided by promised days), followed by promised days and the customer's state. Same-state delivery and parcel weight matter less. The seller's own history was a weak clue. These are patterns, not proven causes.

## Limitations

- Ranking ability is modest (test ROC-AUC 0.66). Late delivery also depends on things unknown at order time, such as weather, strikes and courier problems.
- The percentages are not reliable across periods. In the calmer test months the model's scores were too high (riskiest tenth: predicted 35.5%, actual 11.4%). The app therefore shows bands instead of percentages. The model needs recalibration on recent data.
- The model is weak inside Sao Paulo state: about half of all late orders, but the top-10% rule caught only 6.8% of them.
- It cannot follow month-to-month swings (real late rate from 1.2% to 6.6% in the test months).
- Only about two years of data from one company.
- The label is for the whole order. Items from different sellers may arrive separately. Orders with several sellers are rare (1.3%) and rarely late (1.0%); the reason is unknown.
- Some training orders were delivered after the split date, so their outcome was not yet known on that date. This is a small simplification.
- It was not tested whether acting on a warning actually reduces delays.

## What did not work

- Adding the recent late rate of all orders (14 and 30 days) lowered the scores.
- A random split gave higher scores (LightGBM ROC-AUC 0.74 against 0.67 with a time split), but it is not an honest test of predicting the future.
- Month as a clue misled the tree models, because late months differ between years.

## How to run

    pip install -r requirements.txt
    streamlit run app/app.py

To rebuild the model, download the Olist CSV files into the DATA folder and run the notebooks in order.

## Repository layout

    NOTEBOOK/   notebooks (analysis and packaging)
    DATA/       data files (not included)
    src/        small helper code used by the saved model
    models/     saved model and settings
    app/        Streamlit app
    tests/      automatic checks

Writing ../README.md


# The automatic checks

In [8]:
%%writefile ../tests/test_project.py
import json
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import pytest

BASE = Path(__file__).resolve().parent.parent
sys.path.insert(0, str(BASE / "src"))
from olist_tools import Winsorizer

FORBIDDEN = [
    "order_delivered_customer_date", "order_delivered_carrier_date",
    "delivery_days", "review_score", "delivered_at", "is_late",
]


@pytest.fixture(scope="module")
def settings():
    with open(BASE / "models" / "settings.json") as f:
        return json.load(f)


@pytest.fixture(scope="module")
def model():
    return joblib.load(BASE / "models" / "late_delivery_model.joblib")


def test_no_forbidden_clues(settings):
    clues = settings["numeric_clues"] + settings["category_clues"]
    assert not any(c in clues for c in FORBIDDEN)


def test_bands_are_in_the_right_order(settings):
    assert settings["cut_medium"] < settings["cut_high"]
    rates = {b["band"]: b["late_rate_pct"] for b in settings["band_stats"]}
    assert rates["Low"] < rates["Medium"] < rates["High"]


def test_typical_order_gets_a_valid_score(model, settings):
    cols = settings["numeric_clues"] + settings["category_clues"]
    X = pd.DataFrame([settings["defaults"]])[cols]
    p = model.predict_proba(X)[0, 1]
    assert 0 <= p <= 1


def test_missing_values_do_not_crash(model, settings):
    cols = settings["numeric_clues"] + settings["category_clues"]
    row = dict(settings["defaults"])
    row["distance_km"] = np.nan
    row["main_category"] = np.nan
    X = pd.DataFrame([row])[cols]
    p = model.predict_proba(X)[0, 1]
    assert 0 <= p <= 1


def test_winsorizer_caps_extreme_values():
    X = np.concatenate([np.arange(100), [10_000]]).reshape(-1, 1)
    out = Winsorizer().fit(X).transform(X)
    assert out.max() < 10_000


def test_time_split_has_no_overlap():
    path = BASE / "data" / "full_v2.pkl"
    if not path.exists():
        pytest.skip("data file not available on this computer")
    full = pd.read_pickle(path)
    assert full["order_id"].is_unique
    learn = full[full["order_purchase_timestamp"] < "2018-05-01"]
    test = full[full["order_purchase_timestamp"] >= "2018-05-01"]
    assert learn["order_purchase_timestamp"].max() < test["order_purchase_timestamp"].min()

Writing ../tests/test_project.py


# PYTEST

In [9]:
%pip install pytest


   ---------------------------------------- 0/3 [pluggy]
   -------------------------- ------------- 2/3 [pytest]
   -------------------------- ------------- 2/3 [pytest]
   -------------------------- ------------- 2/3 [pytest]
   -------------------------- ------------- 2/3 [pytest]
   -------------------------- ------------- 2/3 [pytest]
   -------------------------- ------------- 2/3 [pytest]
   -------------------------- ------------- 2/3 [pytest]
   -------------------------- ------------- 2/3 [pytest]
   -------------------------- ------------- 2/3 [pytest]
   ---------------------------------------- 3/3 [pytest]

Note: you may need to restart the kernel to use updated packages.


In [10]:
import sys, subprocess
r = subprocess.run([sys.executable, "-m", "pytest", "../tests", "-q"], capture_output=True, text=True)
print(r.stdout[-3000:])
print(r.stderr[-1000:])

......                                                                   [100%]
6 passed in 9.09s




#  The model card

In [11]:
%%writefile ../MODEL_CARD.md
# Model Card: Olist Late-Delivery Risk

Last updated: 2026-10-30

## Model details

- Type: logistic regression in one scikit-learn pipeline (cleaning, scaling and encoding learned from the learning period only).
- Clues: 33 (29 number clues and 4 category clues), all known when the order is placed. See DATA_DICTIONARY.md.
- Output: a risk score and a risk band (Low, Medium, High). High = riskiest 10% of scores in the learning period, Medium = next 20%, Low = the rest.
- Files: models/late_delivery_model.joblib and models/settings.json.

## Intended use

- Help an operations team decide which orders to look at first, for example "check the High band each week".
- A demo for learning and for a portfolio.

## Not intended for

- Automatic decisions about customers, such as refusing or cancelling an order.
- Exact probabilities. Use the bands.
- Other countries, other companies or periods far from 2017-2018, without checking again.

## Data

- Olist public dataset (Brazil). 96,470 delivered orders with valid dates, 96,203 used after leaving out 267 orders from 2016.
- Learning period: Jan 2017 - Apr 2018 (70,851 orders). Test period: May - Aug 2018 (25,352 orders), used once.
- Target: late = arrived on a later calendar day than the promised day. Late rate was 5.65% in training months, 10.8% in Jan-Apr 2018 and 4.4% in the test months.
- No personal data about customers is used. Only geography (state, ZIP prefix) and order details.

## Performance on the test period

- ROC-AUC 0.665. PR-AUC 0.092 (a lazy model gets 0.044).
- Riskiest 5% of orders: 3.1x the average late rate, 15.7% of late orders found.
- Riskiest 10% of orders: 2.6x the average late rate, 25.9% of late orders found.

| Band | Orders | Really late | Compared with average |
|---|---|---|---|
| Low | 14,637 | 2.88% | 0.65x |
| Medium | 6,015 | 4.64% | 1.05x |
| High | 4,700 | 8.83% | 2.01x |

Low does not mean safe: about 38% of late orders in the test period were in the Low band.

## Where the model works less well (test period)

- Customers in Sao Paulo state (46% of orders, about half of late orders): only 1.9% of their orders were flagged by the top-10% rule and only 6.8% of their late orders were found.
- Customers in Rio de Janeiro state: the model said 15.7% late on average, the real rate was 4.2%, so it over-warns there.
- Customers in Bahia, Espirito Santo and Pernambuco: 53-63% of late orders were found, but many orders were flagged.
- By month: the top-10% rule found 33.6% of late orders in May 2018 but only 16.5% in August 2018.
- Scores were too high in the calmer test months (riskiest tenth: predicted 35.5%, actual 11.4%).

## Other limitations

- Modest ranking ability. Late delivery also depends on weather, strikes and courier problems, which are unknown when the order is placed.
- The label is for the whole order. Items from different sellers may arrive separately.
- Only the first seller of each order is used for seller clues.
- Some training orders were delivered after the split date, so their outcome was not known on that date.
- Clue effects are associations, not causes.
- It was not tested whether acting on a warning reduces delays.

## Proposed monitoring (not yet in place)

- Each week, compare the real late rate of each band with the test table above.
- Watch the share of orders that land in the High band. A big jump means the late rate or the order mix has changed.
- Retrain, and recalibrate the bands, on the most recent months when the bands stop separating late orders (High no longer clearly above Low).

## Ethical notes

- Location is a clue, so the model can treat regions differently. Use it to prioritise checks and extra help, not to reduce service to a region.
- A human should review flagged orders.

Writing ../MODEL_CARD.md


# The data dictionary

In [12]:
%%writefile ../DATA_DICTIONARY.md
# Data Dictionary

The model uses 33 clues. All of them are known when the order is placed. Source tables are from the Olist dataset.

## Target

| Name | Meaning |
|---|---|
| is_late | 1 if the order arrived on a later calendar day than the promised day, otherwise 0. Only delivered orders with valid dates are used. |

## Clues: timing and promise

| Name | Meaning | Source |
|---|---|---|
| purchase_dow | Day of the week of the purchase (0 = Monday) | orders |
| purchase_hour | Hour of the purchase | orders |
| is_weekend | 1 if bought on Saturday or Sunday | orders |
| promise_days | Days from the purchase day to the promised delivery day | orders |
| days_to_first_limit | Days from purchase to the earliest seller hand-over deadline in the order | order items |
| days_to_last_limit | Days from purchase to the latest seller hand-over deadline in the order | order items |
| courier_window_days | Days between the last seller deadline and the promised delivery date | orders, order items |

## Clues: location

| Name | Meaning | Source |
|---|---|---|
| distance_km | Straight-line distance between seller and customer. One median map point per ZIP prefix, points outside Brazil removed | sellers, customers, geolocation |
| km_per_promised_day | distance_km divided by promise_days. How demanding the promise is | derived |
| same_state | 1 if seller and customer are in the same state | derived |
| customer_state | Customer's state | customers |
| seller_state | State of the order's first seller | sellers |

## Clues: order contents

| Name | Meaning | Source |
|---|---|---|
| n_items | Number of items in the order | order items |
| n_products | Number of different products | order items |
| n_sellers | Number of different sellers | order items |
| multi_seller | 1 if the order has more than one seller | derived |
| total_price | Sum of item prices | order items |
| total_freight | Sum of delivery charges | order items |
| freight_ratio | total_freight divided by (total_price + 1) | derived |
| total_weight_g | Sum of product weights in grams | products |
| max_weight_g | Weight of the heaviest product | products |
| total_volume_cm3 | Sum of product volumes (length x height x width) | products |
| density_g_per_cm3 | total_weight_g divided by total_volume_cm3 | derived |
| main_category | Category of the order's first product (Portuguese name; the app shows English) | products, translation |

## Clues: payment

| Name | Meaning | Source |
|---|---|---|
| n_payments | Number of payment records | order payments |
| total_payment | Total value paid | order payments |
| max_installments | Largest number of installments | order payments |
| main_payment_type | Type of the first payment | order payments |

## Clues: recent history (past delivered orders only)

| Name | Meaning |
|---|---|
| seller_prev_orders | Earlier orders of the same seller that were already delivered before this purchase |
| seller_prev_late | How many of those were late |
| seller_late_rate_smooth | (late + 10 x average late rate) divided by (orders + 10). The average comes from the training period only |
| seller_90d_rate | Smoothed late rate of the seller's orders delivered in the 90 days before this purchase |
| route_60d_rate | Smoothed late rate of the same seller state to customer state route in the 60 days before this purchase |

## Never used as clues

- Actual delivery dates, delivery duration, carrier hand-over date, order status.
- Reviews (written after delivery).
- IDs, exact dates and ZIP codes.
- Month of purchase (removed because it misled the tree models).

## How missing values are handled

Inside the model pipeline, with limits learned from the learning period only: numbers are capped at the 1st and 99th percentile, filled with the median (plus a "was missing" flag), and scaled. Missing categories become "unknown". Categories with fewer than 100 orders are grouped together.

Writing ../DATA_DICTIONARY.md


In [13]:
import os
for name in sorted(os.listdir("..")):
    if name.upper() in ("DATA",):
        continue
    print(name)

.pytest_cache
DATA_DICTIONARY.md
MODEL_CARD.md
NOTEBOOK
README.md
app
models
requirements.txt
src
tests
